# Finding the unit algorithmically

Computations of the manuscript.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
using Oscar

In [2]:
G = SL(2, 5)
t = G(matrix(base_ring(G), [1 1; 0 1]))
QG = group_algebra(QQ, G)
n = order(Int, G)
@assert n == 120 && order(t) == 5

(order_G = n, order_t = order(t))

(order_G = 120, order_t = 5)

## 1. Construct the quotient order

Construct the rank-$17$ order $S$ fixed by conjugation by $t$ and inversion, then
$O=S/\mathbb ZN$ of rank $16$ in $A\simeq\mathbb Q^2\times\mathbb Q(\sqrt5)^7$.

In [3]:
orbits = [[one(G)]]
seen = Set(first(orbits))
for g in G
    g in seen && continue
    orbit = unique([t^j * h * t^(-j) for h in (g, inv(g)) for j in 0:4])
    push!(orbits, orbit)
    union!(seen, orbit)
end

B = [sum(QG(g) for g in orbit) for orbit in orbits]
N = sum(B)
@assert length(orbits) == 17
@assert N == sum(QG(g) for g in G)

(number_of_orbits = length(orbits), orbit_sizes = length.(orbits))

(number_of_orbits = 17, orbit_sizes = [1, 10, 10, 1, 10, 10, 10, 10, 10, 2, 2, 10, 10, 10, 2, 2, 10])

In [4]:
p = one(QG) - inv(QQ(n))*N
quotient_basis = [b*p for b in B[2:end]]
A, embed = Hecke._subalgebra(QG, quotient_basis)
@assert dim(A) == 16 && is_commutative(A)

preimages = [has_preimage_with_preimage(embed, b) for b in quotient_basis]
@assert all(first, preimages)
O = order(A, last.(preimages); isbasis=true)

(dimension = dim(A), discriminant = discriminant(O))

(dimension = 16, discriminant = 1343692800000000000000000000000)

In [5]:
fields = Hecke.components(Field, A)
[defining_polynomial(F) for (F, _) in fields]

9-element Vector{QQPolyRingElem}:
 x
 x + 1
 x^2 - 5
 x^2 + 3*x + 1
 x^2 - x - 1
 x^2 - x - 1
 x^2 - x - 1
 x^2 - x - 1
 x^2 - 5*x + 5

## 2. Compute the unit group

Compute explicit generators of $O^\times\simeq(\mathbb Z/2)^2\times\mathbb Z^7$.

In [6]:
M = maximal_order(O)
U, unitmap = unit_group_fac_elem(M; GRH=false)
U

(Z/2)^9 x Z^(7)

In [7]:
K, order_unitmap = unit_group_fac_elem(O; GRH=false)
inclusion = hom(K, U, [unitmap \ order_unitmap(g) for g in gens(K)])
@assert is_injective(inclusion)
K

(Z/2)^2 x Z^(7)

In [8]:
L = hnf(vcat(matrix(inclusion), Hecke.rels(U)))
L = lll(L[1:ngens(U), 1:ngens(U)])
units = [Hecke.elem_in_algebra(O(evaluate(unitmap(U(L[i:i, :])))))
         for i in 1:nrows(L)]

length(units)

16

## 3. Determine the augmentation image

Use `Hecke.closure` to obtain $\bar\varepsilon(O^\times)=\{1,49,71,119\}$ and a unit
with reduced augmentation $49$.

In [9]:
augmentation_mod_n(x) = Int(mod(ZZ(-n*embed(x)[one(G)]), n))
residues = augmentation_mod_n.(units)
residues

16-element Vector{Int64}:
   1
   1
   1
   1
   1
   1
 119
 119
 119
 119
   1
 119
  49
 119
  49
   1

In [10]:
representatives = Dict(Hecke.closure(
    collect(zip(residues, units)),
    (a, b) -> (mod(a[1]*b[1], n), a[2]*b[2]),
    (1, one(A)); eq=(a, b) -> a[1] == b[1]))

augmentation_image = sort!(collect(keys(representatives)))
@assert haskey(representatives, 49)
augmentation_image

4-element Vector{Int64}:
   1
  49
  71
 119

## 4. Lift and verify

Lift the unit and its inverse to augmentation $49$, then verify integrality,
commutation with $t$, and $uv=vu=1+20N$.

In [11]:
x = representatives[49]
u = embed(x) + QQ(49, n)*N
v = embed(inv(x)) + QQ(49, n)*N;

In [12]:
@assert all(isone(denominator(c)) for w in (u, v) for c in coefficients(w))
@assert sum(coefficients(u)) == sum(coefficients(v)) == 49
@assert QG(t)*u == u*QG(t) && QG(t)*v == v*QG(t)
@assert u*v == v*u == one(QG) + 20*N

println("Verified integral lifts: augmentation 49, t-invariant, uv = vu = 1 + 20N.")

Verified integral lifts: augmentation 49, t-invariant, uv = vu = 1 + 20N.
